In [ ]:
import os

PATH_DILIGENCIAS = "../data/Processed/full_MIG.geojson"
PATH_OUTPUT      = "../data/Processed/"

os.makedirs(PATH_OUTPUT, exist_ok=True)

RESOLUCION = 30  # metros
CRS = "EPSG:4326"

In [ ]:
import geopandas as gpd
import pandas as pd
import json



with open(PATH_DILIGENCIAS) as f:
    dilig = json.load(f)
props = [feat['properties'] for feat in dilig['features']]
coords = [feat['geometry']['coordinates'] for feat in dilig['features']]
df_dilig = pd.DataFrame(props)
df_dilig['lon'] = [c[0] for c in coords]
df_dilig['lat'] = [c[1] for c in coords]
df_dilig['Cod_Mun'] = df_dilig['Cod_Mun'].astype(str).str.zfill(5)

gdf_dilig = gpd.GeoDataFrame(
    df_dilig,
    geometry=gpd.points_from_xy(df_dilig['lon'], df_dilig['lat']),
    crs=CRS
)

print(f"  ✓ {len(gdf_dilig)} diligencias")

In [ ]:
import sys, subprocess

subprocess.run([sys.executable, '-m', 'pip', 'install', 'openpyxl'], 
               capture_output=True)

In [ ]:
import pandas as pd

# Leer hoja de víctimas del Excel de Chivolo que ya tienes
from openpyxl import load_workbook

wb = load_workbook("/Users/catalinabernal/Desktop/Uniandes/PEG/Tesis/data/14092022_MIG_MAG_CALD.xlsx", 
                   read_only=True)
ws = wb['Info_Victima_L_Desap']
rows = list(ws.iter_rows(values_only=True))

# Ver encabezados
print("Columnas disponibles:")
for i, col in enumerate(rows[1]):
    if col and any(word in str(col).lower() for word in 
                   ['año', 'fecha', 'desap', 'mes', 'dia']):
        print(f"  [{i}] {col}")

print(f"\nTotal filas: {len(rows)}")

In [ ]:
from openpyxl import load_workbook
import pandas as pd
import numpy as np

ws = wb['Info_Victima_L_Desap']
rows = list(ws.iter_rows(values_only=True))

# Extraer año de desaparición (columna 36)
años_desap = []
for row in rows[2:]:  # saltar encabezados
    año = row[36]
    if año is not None:
        try:
            años_desap.append(int(año))
        except:
            pass

print(f"Registros con año de desaparición: {len(años_desap)}")
print(f"Registros sin año: {len(rows)-2-len(años_desap)}")
print(f"\nDistribución por año:")
s = pd.Series(años_desap)
print(s.value_counts().sort_index())
print(f"\nRango: {min(años_desap)} - {max(años_desap)}")

In [ ]:
import json
import pandas as pd

with open("../data/Processed/full_MIG.geojson") as f:
    data = json.load(f)

props = [f['properties'] for f in data['features']]
df = pd.DataFrame(props)

# Verificar el registro
print(df[df['Año de la diligencia'] == '3013'][
    ['Cod_Mun', 'Año de la diligencia', 'Mes de la diligencia']
])

# Corregir
for feat in data['features']:
    if feat['properties'].get('Año de la diligencia') == '3013.0':
        feat['properties']['Año de la diligencia'] = '2013'
        print("✓ Corregido")

# Guardar geojson corregido
with open("../data/Processed/full_MIG.geojson", 'w') as f:
    json.dump(data, f)

print("✓ Geojson guardado con corrección")

In [ ]:
# Buscar en todos los formatos posibles
año_col = df['Año de la diligencia']
print("Tipos únicos:", año_col.apply(type).unique())
print("\nValores que contienen 3013:")
print(df[año_col.astype(str).str.contains('3013', na=False)][
    ['Cod_Mun', 'Año de la diligencia']
])

In [ ]:
df['Año de la diligencia']

In [ ]:
from pathlib import Path
import pandas as pd

path_equitas = Path("/Volumes/Backup Plus/Equitas/MIG")

# Ver archivos disponibles
excels = list(path_equitas.glob("*.xlsx"))
print(f"Total archivos Excel: {len(excels)}")
for f in sorted(excels):
    print(f"  {f.name}")

In [ ]:
from pathlib import Path
from openpyxl import load_workbook
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

path_equitas = Path("/Volumes/Backup Plus/Equitas/MIG")
excels = list(path_equitas.glob("*.xlsx"))

resultados = []

for excel_path in sorted(excels):
    print(f"\nProcesando: {excel_path.name}")
    
    try:
        wb = load_workbook(excel_path, read_only=True)
        
        # Verificar si tiene la hoja de víctimas
        if 'Info_Victima_L_Desap' not in wb.sheetnames:
            print(f"  ! Sin hoja Info_Victima_L_Desap")
            continue
            
        ws = wb['Info_Victima_L_Desap']
        rows = list(ws.iter_rows(values_only=True))
        
        if len(rows) < 3:
            print(f"  ! Hoja vacía")
            continue
        
        # Encontrar columna de año de desaparición
        encabezados = rows[1]
        col_año = None
        for i, col in enumerate(encabezados):
            if col and str(col).strip() == 'Año de la desaparición':
                col_año = i
                break
        
        if col_año is None:
            print(f"  ! Columna año desaparición no encontrada")
            print(f"    Columnas disponibles: {[c for c in encabezados if c][:10]}")
            continue
        
        # Extraer años
        años = []
        for row in rows[2:]:
            año = row[col_año]
            if año is not None:
                try:
                    años.append(int(float(str(año))))
                except:
                    pass
        
        if años:
            s = pd.Series(años)
            resultados.append({
                'archivo': excel_path.name,
                'n_victimas': len(rows) - 2,
                'n_con_año': len(años),
                'año_min': min(años),
                'año_max': max(años),
                'año_modal': s.mode()[0],
                'años_distribucion': s.value_counts().sort_index().to_dict()
            })
            print(f"  ✓ {len(años)} registros con año")
            print(f"    Rango: {min(años)} - {max(años)}, modal: {s.mode()[0]}")
        else:
            print(f"  ! Sin años válidos")
            
    except Exception as e:
        print(f"  ✗ Error: {e}")

# Resumen
print(f"\n{'='*60}")
print(f"Archivos procesados: {len(resultados)}")
df_resumen = pd.DataFrame(resultados)[
    ['archivo', 'n_victimas', 'n_con_año', 'año_min', 'año_max', 'año_modal']
]
print(df_resumen.to_string(index=False))

In [ ]:
wb = load_workbook("/Volumes/Backup Plus/Equitas/MIG/05102023_MIG_CHIVOLO.xlsx", 
                   read_only=True)
ws = wb['Info_Victima_L_Desap']
rows = list(ws.iter_rows(values_only=True))

print("Todas las columnas con 'año':")
for i, col in enumerate(rows[1]):
    if col and 'año' in str(col).lower():
        print(f"  [{i}] {col}")

In [ ]:
from pathlib import Path
from openpyxl import load_workbook
import pandas as pd
import numpy as np
import json

path_equitas = Path("/Volumes/Backup Plus/Equitas/MIG")

# ============================================================
# 1. EXTRAER COD_REF_EQ + AÑO DE CADA EXCEL
# ============================================================
df_victimas = []

for excel_path in sorted(path_equitas.glob("*.xlsx")):
    if excel_path.name.startswith('._'):
        continue
    
    try:
        wb = load_workbook(excel_path, read_only=True)
        if 'Info_Victima_L_Desap' not in wb.sheetnames:
            continue
        
        ws = wb['Info_Victima_L_Desap']
        rows = list(ws.iter_rows(values_only=True))
        encabezados = rows[1]
        
        # Encontrar columnas clave
        col_cod = None
        col_año = None
        for i, col in enumerate(encabezados):
            if col and 'cod_ref' in str(col).lower():
                col_cod = i
            if col and str(col).strip() == 'Año de la desaparición':
                col_año = i
        
        if col_cod is None or col_año is None:
            print(f"  ! {excel_path.name}: columnas no encontradas")
            continue
        
        for row in rows[2:]:
            cod = row[col_cod]
            año = row[col_año]
            if cod is not None:
                try:
                    año_val = int(float(str(año))) if año else None
                    if año_val and not (1970 <= año_val <= 2023):
                        año_val = None
                except:
                    año_val = None
                df_victimas.append({
                    'COD_REF_EQ': str(cod).strip(),
                    'año_desaparicion': año_val,
                    'archivo': excel_path.name
                })
        
        print(f"✓ {excel_path.name}: {len(rows)-2} víctimas")
        
    except Exception as e:
        print(f"✗ {excel_path.name}: {e}")

df_vic = pd.DataFrame(df_victimas)
print(f"\nTotal víctimas cargadas: {len(df_vic)}")
print(f"Con año: {df_vic['año_desaparicion'].notna().sum()}")
print(f"Sin año: {df_vic['año_desaparicion'].isna().sum()}")

In [ ]:
# ============================================================
# 2. CARGAR GEOJSON Y CRUZAR
# ============================================================
with open("../data/Processed/full_MIG.geojson") as f:
    data = json.load(f)

props = [f['properties'] for f in data['features']]
df_dilig = pd.DataFrame(props)
df_dilig['COD_REF_EQ'] = df_dilig['COD_REF_EQ'].astype(str).str.strip()
df_dilig['Cod_Mun'] = df_dilig['Cod_Mun'].astype(str).str.zfill(5)

print(f"Diligencias en geojson: {len(df_dilig)}")
print(f"Ejemplo COD_REF_EQ geojson: {df_dilig['COD_REF_EQ'].iloc[0]}")
print(f"Ejemplo COD_REF_EQ victimas: {df_vic['COD_REF_EQ'].iloc[0]}")

# Cruce individual
df_merged = df_dilig.merge(
    df_vic[['COD_REF_EQ', 'año_desaparicion', 'archivo']],
    on='COD_REF_EQ',
    how='left'
)

print(f"\nDespués del merge:")
print(f"  Con año individual: {df_merged['año_desaparicion'].notna().sum()}")
print(f"  Sin año individual: {df_merged['año_desaparicion'].isna().sum()}")

# ============================================================
# 3. IMPUTAR FALTANTES CON MODAL DEL CASO
# ============================================================
# Modal por municipio de los que sí tienen año
modal_municipio = (
    df_merged[df_merged['año_desaparicion'].notna()]
    .groupby('Cod_Mun')['año_desaparicion']
    .agg(lambda x: x.mode()[0])
    .to_dict()
)

# Imputaciones manuales para casos sin datos
imputaciones_manuales = {
    '70713': 2001, '70820': 2001, '70429': 2001, '70001': 2001,
    '66456': 2002, '66572': 2002, '18592': 2003,
    '05034': 2003, '05652': 2002, '05854': 2002, '05887': 2002,
}

modal_global = 2002

def asignar_año_final(row):
    # 1. Año individual
    if pd.notna(row['año_desaparicion']):
        return row['año_desaparicion'], 'individual'
    # 2. Modal del municipio
    codigo = row['Cod_Mun']
    if codigo in modal_municipio:
        return modal_municipio[codigo], 'modal_municipio'
    # 3. Imputación manual
    if codigo in imputaciones_manuales:
        return imputaciones_manuales[codigo], 'imputacion_manual'
    # 4. Modal global
    return modal_global, 'modal_global'

resultados = df_merged.apply(asignar_año_final, axis=1)
df_merged['año_asignado'] = [r[0] for r in resultados]
df_merged['fuente_año'] = [r[1] for r in resultados]

# ============================================================
# 4. RESUMEN
# ============================================================
print(f"\nResumen de asignación:")
print(df_merged['fuente_año'].value_counts())

print(f"\nDistribución años asignados:")
print(df_merged['año_asignado'].value_counts().sort_index())

print(f"\nPor municipio — modal asignado:")
resumen_mun = df_merged.groupby('Cod_Mun')['año_asignado'].agg(
    lambda x: x.mode()[0]
).reset_index()
resumen_mun.columns = ['Cod_Mun', 'año_modal_asignado']
print(resumen_mun.sort_values('año_modal_asignado').to_string(index=False))

# Guardar
df_merged[['COD_REF_EQ', 'Cod_Mun', 'año_asignado', 'fuente_año']].to_csv(
    "../data/Processed/anios_asignados.csv", index=False
)
print("\n✓ Guardado en años_asignados.csv")

In [ ]:
# Umbral >= 10, y para casos dispersos usar mediana
modal_municipio_robusto = {}

for cod, grupo in df_merged[df_merged['año_desaparicion'].notna()].groupby('Cod_Mun'):
    años = grupo['año_desaparicion'].dropna()
    if len(años) >= 10:
        # Verificar si la distribución es concentrada o dispersa
        # Concentrada: el modal representa >= 30% de los registros
        modal = años.mode()[0]
        pct_modal = (años == modal).sum() / len(años)
        
        if pct_modal >= 0.3:
            modal_municipio_robusto[cod] = modal
        else:
            # Distribución dispersa — usar mediana
            modal_municipio_robusto[cod] = int(años.median())
            print(f"  {cod}: distribución dispersa → mediana={int(años.median())}")

print("\nModales/medianas robustos (>=10 registros):")
for cod, val in sorted(modal_municipio_robusto.items()):
    n = df_merged[df_merged['Cod_Mun']==cod]['año_desaparicion'].notna().sum()
    print(f"  {cod}: {int(val)} ({n} registros)")

In [ ]:
def asignar_año_final_robusto(row):
    # 1. Año individual
    if pd.notna(row['año_desaparicion']):
        return int(row['año_desaparicion']), 'individual'
    # 2. Modal/mediana robusto del municipio
    codigo = row['Cod_Mun']
    if codigo in modal_municipio_robusto:
        return int(modal_municipio_robusto[codigo]), 'modal_robusto'
    # 3. Imputación manual
    if codigo in imputaciones_manuales:
        return imputaciones_manuales[codigo], 'imputacion_manual'
    # 4. Modal global
    return modal_global, 'modal_global'

resultados = df_merged.apply(asignar_año_final_robusto, axis=1)
df_merged['año_asignado'] = [r[0] for r in resultados]
df_merged['fuente_año'] = [r[1] for r in resultados]

print("Resumen final:")
print(df_merged['fuente_año'].value_counts())

print("\nPor municipio:")
resumen = df_merged.groupby('Cod_Mun')['año_asignado'].agg(
    lambda x: x.mode()[0]
).reset_index()
resumen.columns = ['Cod_Mun', 'año_asignado']
print(resumen.sort_values('año_asignado').to_string(index=False))

# Guardar
df_merged[['COD_REF_EQ', 'Cod_Mun', 
           'año_asignado', 'fuente_año']].to_csv(
    "../data/Processed/anios_asignados.csv", index=False
)
print("\n✓ Guardado")

In [ ]:
import ee
import time

ee.Initialize(project='tu-project-id')

# Municipios con período diferente
municipios_reprocesar = {
    '05543': ('1995-01-01', '2000-12-31'),  # Peñol
    '05628': ('1995-01-01', '2000-12-31'),  # San Carlos
    '05361': ('1999-01-01', '2003-12-31'),  # Ituango
}

def get_ndvi(geometry_ee, inicio, fin):
    return (
        ee.ImageCollection('LANDSAT/LT05/C02/T1_L2')
        .filterDate(inicio, fin)
        .filterBounds(geometry_ee)
        .filter(ee.Filter.lt('CLOUD_COVER', 30))
        .map(lambda img: img.normalizedDifference(['SR_B4', 'SR_B3'])
                           .rename('NDVI'))
        .median()
        .clip(geometry_ee)
    )

for codigo, (inicio, fin) in municipios_reprocesar.items():
    mun = municipios_filtrados[
        municipios_filtrados['MpCodigo'].astype(str).str.zfill(5) == codigo
    ].iloc[0]
    geom_ee = ee.Geometry(mun['geometry'].__geo_interface__)
    ndvi = get_ndvi(geom_ee, inicio, fin)
    
    task = ee.batch.Export.image.toDrive(
        image=ndvi,
        description=f'NDVI_{codigo}_v2',
        folder='landsat_ndvi',
        fileNamePrefix=f'ndvi_{codigo}_v2',
        region=geom_ee,
        scale=30,
        crs='EPSG:4326',
        maxPixels=1e10
    )
    task.start()
    print(f"✓ NDVI_{codigo} ({inicio[:4]}-{fin[:4]}) iniciado")
    time.sleep(1)